# Image Classification Project

## Modules

This notebook demonstrates an image classification workflow, including data loading, preprocessing, model training with various algorithms (SVM, XGBoost, RandomForest), and evaluation.

In [1]:
import os
import pickle

# pyrefly: ignore [missing-import]
import numpy as np
# pyrefly: ignore [missing-import]
from skimage.io import imread
from skimage.transform import resize

from sklearn.model_selection import train_test_split , GroupShuffleSplit

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
# pyrefly: ignore [missing-import]
from xgboost import XGBClassifier

from sklearn.metrics import classification_report , accuracy_score

## Data

In [2]:
data = []
labels = []
groups = []

categories = ['empty', 'not_empty']
input = "data"

for cat_idx , cat in enumerate(categories):
    for file in os.listdir(os.path.join(input,cat)):
        img_path = os.path.join(input , cat , file)

        img = resize( imread(img_path) , (50,50))

        data.append(img.flatten())
        labels.append(cat_idx)
        groups.append(file.split('_')[1].split('.')[0])

data = np.asarray(data)
labels = np.asarray(labels)
groups = np.asarray(groups)

print(len(data),len(labels))

6090 6090


### Data Loading and Preprocessing

## Train / Test split

In [3]:
"""
x_train , x_test , y_train , y_test = train_test_split( data , labels ,
                                                test_size = 0.2,
                                                shuffle = True,
                                                stratify= labels,
                                                random_state=42)
"""

'\nx_train , x_test , y_train , y_test = train_test_split( data , labels ,\n                                                test_size = 0.2,\n                                                shuffle = True,\n                                                stratify= labels,\n                                                random_state=42)\n'

In [4]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(data, labels, groups))

x_train = data[train_idx]
x_test = data[test_idx]
y_train = labels[train_idx]
y_test = labels[test_idx]

## Model Training

### SVMC

### Support Vector Machine (SVC) Training

In [ ]:
svm = SVC()

param = [{'gamma' : [0.01 , 0.001, 0.0001],
        'C' : [1,10,100,1000]
        }]

grid_search = GridSearchCV( svm , param)

grid_search.fit(x_train , y_train)

"svm = SVC()\n\nparam = [{'gamma' : [0.01 , 0.001, 0.0001],\n        'C' : [1,10,100,1000]\n        }]\n\ngrid_search = GridSearchCV( svm , param)\n\ngrid_search.fit(x_train , y_train)"

In [6]:
svm = SVC(C=1, gamma=0.01)

svm.fit(x_train, y_train)

y_pred_svm = svm.predict(x_train)

print("--- SVM Results ---")
print(classification_report(y_train, y_pred_svm))

--- SVM Results ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2371
           1       1.00      1.00      1.00      2469

    accuracy                           1.00      4840
   macro avg       1.00      1.00      1.00      4840
weighted avg       1.00      1.00      1.00      4840



### XGBoost

### XGBoost Classifier Training

In [7]:
xgb = XGBClassifier(
    n_estimators = 500,
    max_depth = 6,
    learning_rate = 0.05,

    # device = "cuda",
    tree_method = "hist",

    subsample = 0.8,
    colsample_bytree = 0.8,

    random_state = 42,
    eval_metric = 'logloss'
)

xgb.fit(x_train, y_train)

y_pred_xgb = xgb.predict(x_train)

print("--- XGBoost Results (Train) ---")
print(classification_report(y_train, y_pred_xgb))

--- XGBoost Results (Train) ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2371
           1       1.00      1.00      1.00      2469

    accuracy                           1.00      4840
   macro avg       1.00      1.00      1.00      4840
weighted avg       1.00      1.00      1.00      4840



### RandomForest

### Random Forest Classifier Training

In [8]:
rfc = RandomForestClassifier(n_estimators=100,max_depth=10,random_state=42,
                            oob_score=True, n_jobs=-1)

rfc.fit(x_train,y_train)

y_rf_pred = rfc.predict(x_train)

print("--- RANDOM FOREST Results (Train) ---")
print(classification_report(y_train, y_rf_pred))


--- RANDOM FOREST Results (Train) ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2371
           1       1.00      1.00      1.00      2469

    accuracy                           1.00      4840
   macro avg       1.00      1.00      1.00      4840
weighted avg       1.00      1.00      1.00      4840



#### OOB

In [9]:
oob_accuracy = rfc.oob_score_
print(f"OOB Score (Global Precision) : {oob_accuracy:.4f}")

OOB Score (Global Precision) : 0.9998


#### Out-of-Bag (OOB) Score for Random Forest

In [10]:
y_pred_xgb = xgb.predict(x_test)
print("---- XGBoost ----")
print(classification_report(y_test,y_pred_xgb))
print("")

y_pred_rf = rfc.predict(x_test)
print("---- RandomForest ----")
print(classification_report(y_test,y_pred_rf))
print("")

score = accuracy_score ( svm.predict(x_test) , y_test )
print("---- SVM ---- : ",score)

---- XGBoost ----
              precision    recall  f1-score   support

           0       0.97      0.82      0.89       674
           1       0.82      0.97      0.89       576

    accuracy                           0.89      1250
   macro avg       0.90      0.90      0.89      1250
weighted avg       0.90      0.89      0.89      1250


---- RandomForest ----
              precision    recall  f1-score   support

           0       0.99      0.84      0.91       674
           1       0.84      0.99      0.91       576

    accuracy                           0.91      1250
   macro avg       0.91      0.91      0.91      1250
weighted avg       0.92      0.91      0.91      1250


---- SVM ---- :  0.9224


## Model Evaluation on Test Data

## loading the model

In [11]:
pickle.dump(svm , open("./model.p" , "wb"))

This section saves the trained SVM model for later use.

In [12]:
busy_A = "data_2/A/busy"
free_A = "data_2/A/free"

data_busy_A  , data_free_A = [] , []

for cat in [busy_A , free_A] :
    if cat == busy_A :
        for file in os.listdir(busy_A):

            img_path = os.path.join(busy_A , file)

            img = resize( imread(img_path) , (50,50))
            data_busy_A.append(img.flatten())

    if cat == free_A :
        for file in os.listdir(free_A):

            img_path = os.path.join(free_A , file)

            img = resize( imread(img_path) , (50,50))
            data_free_A.append(img.flatten())

data_busy_A = np.asarray(data_busy_A)
data_free_A = np.asarray(data_free_A)

print(len(data_busy_A))
print(len(data_free_A))

3621
2550


### Loading and Predicting on Data from Set A

In [13]:
pred =svm.predict(data_free_A)
counts = np.bincount(pred)
print(f"Empty spaces (0): {counts[0]}")
print(f"Occupied spaces (1): {counts[1]}")

Empty spaces (0): 1454
Occupied spaces (1): 1096


#### SVM Predictions on `data_free_A`

In [ ]:
pred = svm.predict(data_busy_A)

counts = np.bincount(pred)
print(f"Empty spaces (0): {counts[0]}")
print(f"Occupied spaces (1): {counts[1]}")


#### SVM Predictions on `data_busy_A`

In [ ]:
pred = xgb.predict(data_free_A)
counts = np.bincount(pred)
print(f"Empty spaces (0): {counts[0]}")
print(f"Occupied spaces (1): {counts[1]}")

#### XGBoost Predictions on `data_free_A`

In [ ]:
pred = xgb.predict(data_busy_A)
counts = np.bincount(pred)
print(f"Empty spaces (0): {counts[0]}")
print(f"Occupied spaces (1): {counts[1]}")

#### XGBoost Predictions on `data_busy_A`

In [ ]:
pred = rfc.predict(data_free_A)
counts = np.bincount(pred)
print(f"Empty spaces (0): {counts[0]}")
print(f"Occupied spaces (1): {counts[1]}")

#### Random Forest Predictions on `data_free_A`

In [ ]:
pred = rfc.predict(data_busy_A)
counts = np.bincount(pred)
print(f"Empty spaces (0): {counts[0]}")
print(f"Occupied spaces (1): {counts[1]}")

#### Random Forest Predictions on `data_busy_A`

# B

### Loading and Predicting on Data from Set B

In [ ]:
busy_B = "data_2/B/busy"
free_B = "data_2/B/free"

data_busy_B , data_free_B= [] , []

for cat in [busy_B, free_B] :
    if cat == busy_B:
        for file in os.listdir(busy_B):

            img_path = os.path.join(busy_B, file)

            img = resize( imread(img_path) , (50,50))
            data_busy_B.append(img.flatten())

    if cat == free_B:
        for file in os.listdir(free_B):

            img_path = os.path.join(free_B, file)

            img = resize( imread(img_path) , (50,50))
            data_free_B.append(img.flatten())

data_busy_B= np.asarray(data_busy_B)
data_free_B= np.asarray(data_free_B)

print(len(data_busy_B))
print(len(data_free_B))

4781
1632


This section loads and prepares data from 'Set B' for prediction.

In [ ]:
pred = rfc.predict(data_free_B)
counts = np.bincount(pred)
print(f"Empty spaces (0): {counts[0]}")
print(f"Occupied spaces (1): {counts[1]}")

Empty spaces (0): 588
Occupied spaces (1): 1044


#### Random Forest Predictions on `data_free_B`

In [ ]:
pred = rfc.predict(data_busy_B)
counts = np.bincount(pred)
print(f"Empty spaces (0): {counts[0]}")
print(f"Occupied spaces (1): {counts[1]}")

Empty spaces (0): 1353
Occupied spaces (1): 3428


#### Random Forest Predictions on `data_busy_B`

In [ ]:
labels = [1] * len(data_busy_A) + [0] * len(data_free_A)

labels += [1] * len(data_busy_B) + [0] * len(data_free_B)

full_data = np.concatenate([data_busy_A, data_free_A, data_busy_B, data_free_B], axis=0)

### Predictions on Combined Data (Data A and Data B)

In [ ]:
pred = rfc.predict(full_data)
print("---- RandomForest ----")
print(classification_report(labels,pred))
print("")

---- RandomForest ----
              precision    recall  f1-score   support

           0       0.40      0.30      0.34      4182
           1       0.69      0.78      0.73      8402

    accuracy                           0.62     12584
   macro avg       0.55      0.54      0.54     12584
weighted avg       0.59      0.62      0.60     12584




#### Random Forest Predictions on `full_data`

In [ ]:
pred = svm.predict(full_data)
print("---- SVM ----")
print(classification_report(labels,pred))
print("")

---- SVM ----
              precision    recall  f1-score   support

           0       0.67      0.53      0.60      4182
           1       0.79      0.87      0.83      8402

    accuracy                           0.76     12584
   macro avg       0.73      0.70      0.71     12584
weighted avg       0.75      0.76      0.75     12584




#### SVM Predictions on `full_data`

In [ ]:
pred = xgb.predict(full_data)
print("---- XGBoost ----")
print(classification_report(labels,pred))
print("")

---- XGBoost ----
              precision    recall  f1-score   support

           0       0.40      0.41      0.40      4182
           1       0.70      0.70      0.70      8402

    accuracy                           0.60     12584
   macro avg       0.55      0.55      0.55     12584
weighted avg       0.60      0.60      0.60     12584




#### XGBoost Predictions on `full_data`

In [ ]:
x_train , x_test , y_train , y_test = train_test_split( full_data , labels ,
                                                test_size = 0.2,
                                                shuffle = True,
                                                stratify= labels,
                                                random_state=42)

## Retraining and Re-evaluation with Combined Data

In [ ]:
rfc = RandomForestClassifier(n_estimators=100,max_depth=10,random_state=42,
                            oob_score=True, n_jobs=-1)

rfc.fit(x_train,y_train)

y_rf_pred = rfc.predict(x_train)

print("--- RANDOM FOREST Results (Train) ---")
print(classification_report(y_train, y_rf_pred))


--- RANDOM FOREST Results (Train) ---
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      3346
           1       0.99      1.00      0.99      6721

    accuracy                           0.99     10067
   macro avg       0.99      0.99      0.99     10067
weighted avg       0.99      0.99      0.99     10067



### Random Forest Retraining on Combined Data

In [ ]:
pred = rfc.predict(x_test)
print("---- RandomForest ----")
print(classification_report(y_test,pred))
print("")

---- RandomForest ----
              precision    recall  f1-score   support

           0       0.98      0.97      0.97       836
           1       0.99      0.99      0.99      1681

    accuracy                           0.98      2517
   macro avg       0.98      0.98      0.98      2517
weighted avg       0.98      0.98      0.98      2517




### Random Forest Evaluation on Combined Test Data

## Summary of Classification Results

This notebook evaluates the performance of three different classification models: Support Vector Machine (SVM), XGBoost Classifier, and Random Forest Classifier on both the initial test data and then after retraining on combined data.

### Initial Test Data Evaluation (on `x_test`, `y_test` from initial split):

*   **XGBoost:** Achieved an accuracy of **0.89**. It showed slightly better recall for class 1 (occupied spaces) but lower precision compared to class 0 (empty spaces).
*   **Random Forest:** Achieved an accuracy of **0.91**. Similar to XGBoost, it performed well, showing a good balance in precision and recall for both classes.
*   **SVM:** Achieved an accuracy of **0.9224**. This indicates SVM performed slightly better than XGBoost and Random Forest on the initial test set.

### Predictions on Data Set A (`data_free_A`, `data_busy_A`):

*   **SVM:**
    *   `data_free_A` (expected empty): Predicted 1454 empty and 1096 occupied.
    *   `data_busy_A` (expected busy): Predicted 378 empty and 3243 occupied.
*   **XGBoost:**
    *   `data_free_A` (expected empty): Predicted 750 empty and 1800 occupied.
    *   `data_busy_A` (expected busy): Predicted 1007 empty and 2614 occupied.
*   **Random Forest:**
    *   `data_free_A` (expected empty): Predicted 674 empty and 1876 occupied.
    *   `data_busy_A` (expected busy): Predicted 528 empty and 3093 occupied.

### Predictions on Data Set B (`data_free_B`, `data_busy_B`):

*   **Random Forest:**
    *   `data_free_B` (expected empty): Predicted 588 empty and 1044 occupied.
    *   `data_busy_B` (expected busy): Predicted 1353 empty and 3428 occupied.

### Predictions on Combined Data (`full_data`):

When making predictions on the combined `full_data` (without retraining on it first):

*   **Random Forest:** Achieved an accuracy of **0.62**. The performance dropped significantly, especially for class 0 (empty spaces) with a precision of 0.40 and recall of 0.30.
*   **SVM:** Achieved an accuracy of **0.76**. SVM performed better than Random Forest on the combined data without retraining, showing more balanced precision and recall for both classes.
*   **XGBoost:** Achieved an accuracy of **0.60**. Similar to Random Forest, its performance also decreased on the combined data.

### Retraining and Re-evaluation with Combined Data:

After retraining the **Random Forest Classifier** on the `full_data`:

*   **Train Data:** Achieved a high accuracy of **0.99**. Both precision and recall for both classes were very high, indicating strong learning on the combined training set.
*   **Test Data (from combined split):** Achieved an accuracy of **0.98**. The model demonstrated excellent performance on unseen test data from the combined dataset, with high precision and recall for both 'empty' and 'occupied' classes. This highlights the importance of training on a representative dataset when new data distributions are introduced.